In [5]:
import arcpy
import numpy as np
import random
import math
import os

# 设置工作空间
arcpy.env.workspace = r"D:\ArcGIS\sunleigangPro\模拟退火"
arcpy.env.overwriteOutput = True

# 输入栅格文件
input_raster = "最终ok.tif"

# 输出点要素类
output_points = "output_points.shp"

# 点的高度
point_height = 500  # 单位：米

# 视域范围
view_distance = 200  # 单位：米

# 目标可见区域比例
target_coverage = 0.9  # 90%

# 模拟退火算法参数
Imax = 300  # 每个温度下的最大迭代次数100
theta = 0.9  # 温度衰减系数0.99
r_threshold = 0.5  # 随机数阈值
R = view_distance/2  # 点之间的最小距离
amin = target_coverage  # 最小覆盖率阈值

# 将栅格数据加载到 NumPy 数组中
def raster_to_array(raster):
    raster_array = arcpy.RasterToNumPyArray(raster, nodata_to_value=0)
    return raster_array

# 计算栅格的有效面积
def calculate_raster_area(raster):
    cell_size = float(arcpy.GetRasterProperties_management(raster, "CELLSIZEX").getOutput(0))
    raster_array = raster_to_array(raster)
    valid_cell_count = np.count_nonzero(raster_array)
    raster_area = valid_cell_count * (cell_size ** 2)
    return raster_area

# 获取栅格的有效区域多边形
def get_raster_domain(raster):
    # 将栅格的有效区域转换为多边形
    domain_polygon = os.path.join(arcpy.env.workspace, "raster_domain.shp")
    arcpy.RasterDomain_3d(raster, domain_polygon, "POLYGON")
    return domain_polygon

# 计算初始点的数量
def calculate_initial_points(raster_area, view_distance, target_coverage):
    # 每个点的视域范围面积（圆形区域）
    view_area = math.pi * (R ** 2)  # 视域范围为圆形，半径为R
    # 初始点数量 = 栅格面积 * 可见比例 / 视域范围面积
    initial_points = int(raster_area * target_coverage / view_area)
    return initial_points

# Poisson Disk 采样算法
def poisson_disk_sampling(width, height, radius, k=30):
    # 初始化网格和点列表
    cell_size = radius / np.sqrt(2)
    grid_width = int(np.ceil(width / cell_size))
    grid_height = int(np.ceil(height / cell_size))
    grid = [[None for _ in range(grid_width)] for _ in range(grid_height)]
    points = []
    active = []
    
    # 设置随机数种子，确保每次生成的点分布不相同
    random.seed(time.time())
    # 添加初始点
    initial_point = (random.uniform(0, width), random.uniform(0, height))
    points.append(initial_point)
    active.append(initial_point)
    grid[int(initial_point[1] / cell_size)][int(initial_point[0] / cell_size)] = initial_point
    
    # 生成点
    while active:
        random_index = random.randint(0, len(active) - 1)
        point = active[random_index]
        found = False
        
        for _ in range(k):
            angle = random.uniform(0, 2 * np.pi)
            distance = random.uniform(radius, 2 * radius)
            new_point = (point[0] + distance * np.cos(angle), point[1] + distance * np.sin(angle))
            
            if 0 <= new_point[0] < width and 0 <= new_point[1] < height:
                grid_x = int(new_point[0] / cell_size)
                grid_y = int(new_point[1] / cell_size)
                valid = True
                
                for i in range(max(0, grid_x - 2), min(grid_width, grid_x + 3)):
                    for j in range(max(0, grid_y - 2), min(grid_height, grid_y + 3)):
                        neighbor = grid[j][i]
                        if neighbor and np.hypot(neighbor[0] - new_point[0], neighbor[1] - new_point[1]) < radius:
                            valid = False
                            break
                    if not valid:
                        break
                
                if valid:
                    points.append(new_point)
                    active.append(new_point)
                    grid[grid_y][grid_x] = new_point
                    found = True
        
        if not found:
            active.pop(random_index)
    
    return points

# 生成随机点（确保点在栅格的有效区域内，且均匀分布）
def generate_random_points(num_points, domain_polygon):
    # 获取有效区域的范围
    extent = arcpy.Describe(domain_polygon).extent
    width = extent.XMax - extent.XMin
    height = extent.YMax - extent.YMin
    
    # 计算最小距离
    area = width * height
    # min_distance = math.sqrt(area / num_points)
    min_distance = min(math.floor(width/(math.sqrt(num_points))),math.floor(height/(math.sqrt(num_points))))
    # 使用 Poisson Disk 采样生成均匀点
    points = []
    while len(points) < num_points:
        new_points = poisson_disk_sampling(width, height, min_distance)
        
        # 将点转换为实际坐标
        actual_points = [(x + extent.XMin, y + extent.YMin) for x, y in new_points]
        
        # 筛选出位于有效区域内的点
        with arcpy.da.SearchCursor(domain_polygon, ["SHAPE@"]) as cursor:
            for row in cursor:
                polygon = row[0]
                for point in actual_points:
                    if polygon.contains(arcpy.PointGeometry(arcpy.Point(point[0], point[1]))):
                        points.append(point)
        
        # 如果生成的点足够，退出循环
        if len(points) >= num_points:
            break
    
    return points[:num_points]  # 返回指定数量的点

# 计算视域覆盖率
def calculate_coverage(points):
    # 获取输入栅格的坐标系
    spatial_ref = arcpy.Describe(input_raster).spatialReference
    
    # 删除旧的 temp_points.shp 文件
    temp_points = os.path.join(arcpy.env.workspace, "temp_points.shp")
    if arcpy.Exists(temp_points):
        arcpy.management.Delete(temp_points)
    
    # 创建点要素类时设置坐标系
    arcpy.CreateFeatureclass_management(
        arcpy.env.workspace,
        "temp_points.shp",
        "POINT",
        spatial_reference=spatial_ref  # 设置坐标系
    )
    
    # 插入点数据
    with arcpy.da.InsertCursor("temp_points.shp", ["SHAPE@XY"]) as cursor:
        for point in points:
            if not math.isnan(point[0]) and not math.isnan(point[1]):  # 检查坐标是否有效
                cursor.insertRow([point])
    
    # 执行视域分析
    viewshed_result = arcpy.sa.Viewshed2(
        in_raster=input_raster,
        in_observer_features="temp_points.shp",
        out_agl_raster=None,
        analysis_type="FREQUENCY",
        vertical_error="0 Meters",
        out_observer_region_relationship_table=None,
        refractivity_coefficient=0.13,
        surface_offset="0 Meters",
        observer_elevation=point_height,
        observer_offset="1 Meters",
        inner_radius=None,
        inner_radius_is_3d="GROUND",
        outer_radius=view_distance / 2,
        outer_radius_is_3d="GROUND",
        horizontal_start_angle=0,
        horizontal_end_angle=360,
        vertical_upper_angle=90,
        vertical_lower_angle=-90,
        analysis_method="ALL_SIGHTLINES",
        analysis_target_device="GPU_THEN_CPU"
    )
    
    # 将结果转换为 NumPy 数组
    viewshed_array = raster_to_array(viewshed_result)
    
    # 计算可见区域面积
    unique_values, counts = np.unique(viewshed_array, return_counts=True)
    cell_size = float(arcpy.GetRasterProperties_management(input_raster, "CELLSIZEX").getOutput(0))
    visible_area = 0
    for value, count in zip(unique_values, counts):
        if value > 0:  # 视域值大于0表示可见区域
            visible_area += count * (cell_size ** 2)
    
    # 计算覆盖率
    coverage = visible_area / raster_area
    print(f"当前覆盖率为: {coverage}")
    return coverage


# 动态调整点数以满足目标覆盖率（二分法）
def optimize_coverage():
    # 初始点数
    num_points = calculate_initial_points(raster_area, view_distance, target_coverage)
    print(f"初始点数量: {num_points}")
    Nmin = num_points
    # 初始运行算法
    points = generate_random_points(Nmin, domain_polygon)
    current_points = points
    current_coverage = calculate_coverage(current_points)
    best_points = current_points
    best_coverage = current_coverage
    
    # 如果初始点数不满足覆盖率，增加点数
    if best_coverage < target_coverage:
        Nmin = Nmin + 1
        Nmax = 3 * num_points
        print(f"初始覆盖率 {best_coverage * 100}% 未达到目标，增加点数范围至 [{Nmin}, {Nmax}]")
        
    # 二分法搜索
    while Nmax != Nmin:
        # 计算中间点数
        N = (Nmax + Nmin) // 2  # 向下取整
        print(f"尝试点数: {N}")
        i = 0
        best_coverage = 0
        while(i<Imax):
            new_points = generate_random_points(N, domain_polygon)
            new_coverage = calculate_coverage(new_points)
            # 更新最优解
            if new_coverage > best_coverage:
                best_points = new_points
                best_coverage = new_coverage
            if best_coverage > target_coverage:
                Nmax = N
                print(f"当前覆盖率 {best_coverage * 100}% 达到目标，减少点数范围至 [{Nmin}, {Nmax}]")
                best_feasible_points = best_points
                best_feasible_coverage = best_coverage
                break
            i += 1
        # 根据覆盖率调整搜索范围
        if best_coverage < target_coverage:
            Nmin = N + 1
            print(f"当前覆盖率 {best_coverage * 100}% 未达到目标，增加点数范围至 [{Nmin}, {Nmax}]")

    return  best_feasible_points, best_feasible_coverage

# 主程序
if __name__ == "__main__":
    # 获取栅格的有效面积
    raster_area = calculate_raster_area(input_raster)
    print(f"栅格的有效面积: {raster_area} 平方米")
    
    # 获取栅格的有效区域多边形
    domain_polygon = get_raster_domain(input_raster)
    
    # 运行动态调整点数的优化算法
    best_feasible_points, best_feasible_coverage = optimize_coverage()
    
    # 输出结果
    print(f"最优点的数量: {len(best_feasible_points)}")
    print(f"最优点的位置: {best_feasible_points}")
    print(f"可见区域覆盖率: {best_feasible_coverage * 100}%")
    
    # 删除旧的 output_points.shp 文件
    if arcpy.Exists(output_points):
        arcpy.management.Delete(output_points)
    
    # 保存最优点的位置到点要素类
    spatial_ref = arcpy.Describe(input_raster).spatialReference
    arcpy.CreateFeatureclass_management(
        arcpy.env.workspace,
        output_points,
        "POINT",
        spatial_reference=spatial_ref  # 设置坐标系
    )
    
    # 添加 X 和 Y 坐标字段
    arcpy.management.AddField(output_points, "X", "DOUBLE")
    arcpy.management.AddField(output_points, "Y", "DOUBLE")
    
    # 插入点数据
    with arcpy.da.InsertCursor(output_points, ["SHAPE@XY", "X", "Y"]) as cursor:
        for point in best_feasible_points:
            if not math.isnan(point[0]) and not math.isnan(point[1]):  # 检查坐标是否有效
                cursor.insertRow([point, point[0], point[1]])
    
    # 将生成的要素类加载到地图中
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    map = aprx.listMaps()[0]  # 获取第一个地图
    map.addDataFromPath(os.path.join(arcpy.env.workspace, output_points))
    
    print("处理完成！")

栅格的有效面积: 572800.0 平方米
初始点数量: 16
当前覆盖率为: 0.5277583798882681
当前覆盖率为: 0.5115223463687151
当前覆盖率为: 0.5171089385474861
当前覆盖率为: 0.5267108938547486
当前覆盖率为: 0.5380586592178771
当前覆盖率为: 0.5013966480446927
当前覆盖率为: 0.5516759776536313
当前覆盖率为: 0.4795740223463687
当前覆盖率为: 0.5691340782122905
当前覆盖率为: 0.5013966480446927
当前覆盖率为: 0.5089036312849162
当前覆盖率为: 0.5260125698324022
当前覆盖率为: 0.544518156424581
当前覆盖率为: 0.5560405027932961
当前覆盖率为: 0.5078561452513967
当前覆盖率为: 0.5221717877094972
当前覆盖率为: 0.5457402234636871
当前覆盖率为: 0.5020949720670391
当前覆盖率为: 0.5001745810055865
当前覆盖率为: 0.5034916201117319
当前覆盖率为: 0.5174581005586593
当前覆盖率为: 0.5052374301675978
当前覆盖率为: 0.5099511173184358
当前覆盖率为: 0.5590083798882681
当前覆盖率为: 0.5354399441340782
当前覆盖率为: 0.5288058659217877
当前覆盖率为: 0.5174581005586593
当前覆盖率为: 0.6038756983240223
当前覆盖率为: 0.5045391061452514
当前覆盖率为: 0.4886522346368715
当前覆盖率为: 0.5125698324022346
当前覆盖率为: 0.5309008379888268
当前覆盖率为: 0.5082053072625698
当前覆盖率为: 0.5385824022346368
当前覆盖率为: 0.5298533519553073
当前覆盖率为: 0.51169692737430

当前覆盖率为: 0.8526536312849162
当前覆盖率为: 0.8481145251396648
当前覆盖率 89.94413407821229% 未达到目标，增加点数范围至 [36, 36]
最优点的数量: 36
最优点的位置: [(333551.8805112004, 3804547.3381152903), (333533.58764249587, 3804417.997843069), (333691.66346118564, 3804235.1561436877), (333458.51888271153, 3804706.5517945243), (333374.0118074839, 3804467.306225398), (333425.25174132583, 3804588.0270625064), (333257.11575794296, 3804649.6890075654), (333497.7641121392, 3804255.455472601), (333455.47266174457, 3804061.830063514), (333330.68304501456, 3804151.721423874), (333375.92170028226, 3804325.1351003903), (333204.11268487683, 3804050.70591821), (333338.5228164956, 3803942.8022728637), (333248.15800495225, 3804255.696097525), (333120.18783290294, 3804160.800327022), (333069.4039515288, 3804046.9621191057), (333212.9990591895, 3803872.265334447), (333071.18308379163, 3804302.0823876634), (333191.1792771002, 3804365.3380676066), (333501.5473779108, 3803893.1208056654), (333232.0547709786, 3804486.6290661595), (333656.4019463